#request resources + use analy environment


salloc --partition test --time 0-04:00 --mem 20gb
module load python
mamba activate analy
cd /n/home07/than157/desktop/done-large_projects/learn-better/evolm/finetune/llama-factory/
jupyter notebook --no-browser --ip=0.0.0.0 --port=8888

In [1]:
from datasets import load_dataset
import pandas as pd
import json
from tqdm import tqdm
import numpy as np

## load data

In [2]:
#dongboklee/MMLU-Pro-Llama-3.1-70B-Instruct-CoT
#test split (n=131k) --> our training set
#val split (n=840 --> 70 unique questions) --> our test set

#load training set
train_dataset = load_dataset(path="dongboklee/MMLU-Pro-Llama-3.1-70B-Instruct-CoT", split="test")

df = train_dataset.to_pandas()
print("# samples:", df.shape[0])
print("# of unique questions:", df['question'].nunique())

df.head()


# samples: 130811
# of unique questions: 10532


,question_id,question,options,answer,answer_index,category,src,cot_id,cot_content
0,70,"Typical advertising regulatory bodies suggest,...","[Safe practices, Fear, Jealousy, Trivial, Unsa...",I,8,business,ori_mmlu-business_ethics,0,1. Advertising regulatory bodies typically aim...
1,70,"Typical advertising regulatory bodies suggest,...","[Safe practices, Fear, Jealousy, Trivial, Unsa...",I,8,business,ori_mmlu-business_ethics,1,"First, typical advertising regulatory bodies a..."
2,70,"Typical advertising regulatory bodies suggest,...","[Safe practices, Fear, Jealousy, Trivial, Unsa...",I,8,business,ori_mmlu-business_ethics,2,1. Advertising regulatory bodies typically aim...
3,70,"Typical advertising regulatory bodies suggest,...","[Safe practices, Fear, Jealousy, Trivial, Unsa...",I,8,business,ori_mmlu-business_ethics,3,Typical advertising regulatory bodies aim to e...
4,70,"Typical advertising regulatory bodies suggest,...","[Safe practices, Fear, Jealousy, Trivial, Unsa...",I,8,business,ori_mmlu-business_ethics,4,"First, typical advertising regulatory bodies a..."


In [3]:
#load test set
test_dataset = load_dataset(path="dongboklee/MMLU-Pro-Llama-3.1-70B-Instruct-CoT", split="validation")
test_df = test_dataset.to_pandas()
print(test_df.shape)
print("# of unique questions:", test_df['question'].nunique())

test_df.head()

(840, 9)
# of unique questions: 67


,question_id,question,options,answer,answer_index,category,src,cot_id,cot_content
0,0,The symmetric group $S_n$ has $\n\factorial{n}...,"[0, 30, 3, 10, 12, 50, 2, 100, 20, 5]",A,0,math,cot_lib-abstract_algebra,0,"We are given a problem with two parts, but the..."
1,0,The symmetric group $S_n$ has $\n\factorial{n}...,"[0, 30, 3, 10, 12, 50, 2, 100, 20, 5]",A,0,math,cot_lib-abstract_algebra,1,"To find the characteristic of the ring 2Z, we ..."
2,0,The symmetric group $S_n$ has $\n\factorial{n}...,"[0, 30, 3, 10, 12, 50, 2, 100, 20, 5]",A,0,math,cot_lib-abstract_algebra,2,We are asked to find the characteristic of the...
3,0,The symmetric group $S_n$ has $\n\factorial{n}...,"[0, 30, 3, 10, 12, 50, 2, 100, 20, 5]",A,0,math,cot_lib-abstract_algebra,3,The symmetric group $S_n$ indeed has $n!$ elem...
4,0,The symmetric group $S_n$ has $\n\factorial{n}...,"[0, 30, 3, 10, 12, 50, 2, 100, 20, 5]",A,0,math,cot_lib-abstract_algebra,4,We're given a question about the characteristi...


In [4]:
#df = train df 
#test_df = test df 

## process training data

##### check whether all answers in cot response are correct -- no!

In [5]:
def extract_from_answer_is(c):
    split = c.split("answer is")
    if len(split) > 1:
        answer = split[-1].strip()
        if ":" in answer:
            answer = answer.split(":")[-1].strip()
        answer = answer.rstrip(".,;!?")
        return answer
    else:
        return None

In [6]:
df['extracted_answer'] = df.apply(lambda row: extract_from_answer_is(row['cot_content']), axis=1)

# number of incorrect answers
print('# incorrect answers: ', (df['extracted_answer'] != df['answer']).sum() )

# incorrect answers:  699


In [7]:
#examine incorrect answers
rows_w_incorrect_answer = df[df['extracted_answer'] != df['answer']]

for idx, row in rows_w_incorrect_answer.iterrows():
    print('Question: ', row['question'])
    print('COT: ', row['cot_content'][-100:])
    print('TrueAnswer: ', row['answer'])
    print('Extracted Answer: ', row['extracted_answer'])
    print('-' * 40)

Question:  Assume the following model (from the preceding problem). Y = C + I + G C = 100 + 0.6Y I = 0.2Y - 50i M_D = 0.25Y - 30i M_s = 65 G = 100 whose equilibrium level was found to be 500. Suppose that full employment level of income is 600, so that the desired change is 100. If the money supply is held constant, what change in govern-ment spending will be required to close the deflationary gap?
COT:  f the 40 is multiplied by 1.525 (which is 1/ 0.65) this is 61. So the answer should be 61.5, or (F).
TrueAnswer:  F
Extracted Answer:  None
----------------------------------------
Question:  Suppose the demand curve for oPads is given by $p=\frac{500-x}{10}, What is the elasticity value of this demand function.
COT:  s given, we can see that the only option that matches our calculations for a specific x is:

I. -1.5
TrueAnswer:  I
Extracted Answer:  None
----------------------------------------
Question:  Tim is a salesman who earns a guaranteed salary of $4800/year plus 4% of all sal

Above: even though some of the incorrect answers are actually correct (but just don't end in "The answer is A." format), there are too many diverse formats to account for, so just dropped them all.

##### keep only rows with correct extracted answers

In [8]:
df = df[df['extracted_answer'] == df['answer']].reset_index(drop=True)
print(df.shape)
print(df.question.nunique())

(130112, 10)
10503


In [9]:
df.head()

,question_id,question,options,answer,answer_index,category,src,cot_id,cot_content,extracted_answer
0,70,"Typical advertising regulatory bodies suggest,...","[Safe practices, Fear, Jealousy, Trivial, Unsa...",I,8,business,ori_mmlu-business_ethics,0,1. Advertising regulatory bodies typically aim...,I
1,70,"Typical advertising regulatory bodies suggest,...","[Safe practices, Fear, Jealousy, Trivial, Unsa...",I,8,business,ori_mmlu-business_ethics,1,"First, typical advertising regulatory bodies a...",I
2,70,"Typical advertising regulatory bodies suggest,...","[Safe practices, Fear, Jealousy, Trivial, Unsa...",I,8,business,ori_mmlu-business_ethics,2,1. Advertising regulatory bodies typically aim...,I
3,70,"Typical advertising regulatory bodies suggest,...","[Safe practices, Fear, Jealousy, Trivial, Unsa...",I,8,business,ori_mmlu-business_ethics,3,Typical advertising regulatory bodies aim to e...,I
4,70,"Typical advertising regulatory bodies suggest,...","[Safe practices, Fear, Jealousy, Trivial, Unsa...",I,8,business,ori_mmlu-business_ethics,4,"First, typical advertising regulatory bodies a...",I


## remove some samples from training data to use as test data

##### whether to sample using question_ids or question -- use question because it turns out some questions have multiple question_id's

In [10]:
print('# unique question_ids:', df['question_id'].nunique())
print('# unique questions:', df['question'].nunique())

# unique question_ids: 10688
# unique questions: 10503


In [11]:
unique_pairs = df[['question', 'question_id']].drop_duplicates()
print(unique_pairs.shape)

(10688, 2)


In [12]:
#question_id's that map to multiple questions
question_ids_multiple_questions = (
    df.groupby('question_id')['question']
      .nunique()
      .reset_index(name='num_questions')
      .query('num_questions > 1')
)

question_ids_multiple_questions

,question_id,num_questions


In [13]:
#question that has multiple question_id values
questions_multiple_ids = (
    df.groupby('question')['question_id']
      .nunique()
      .loc[lambda x: x > 1]
      .sort_values(ascending=False)
)

questions_multiple_ids

question
Which of the following is true?                                                                                                                                                                                                                                                                     5
A biologist deals with things on a microscopic level. To A biologist deals with things on a microscopic level. To describe cellular dimensions and the amount of materials present at the cellular level, units of an appropriately small size are needed. What are these units of measurements?    3
What is the difference between a kinesis and a taxis?                                                                                                                                                                                                                                               3
IfDNAaseis added to a bacterial cell, the DNA is hydrolyzed , the cell cannot make any more proteins and even

##### remove subset from training set

In [14]:
#get list of unique questions in training set
unique_questions = df['question'].unique()

#randomly select 500 questions -- these questions will be removed from training set, and added to test set
rng = np.random.default_rng(42)
n_samples_move_from_train_to_test = 500
selected_questions = rng.choice(unique_questions, size=n_samples_move_from_train_to_test, replace=False) 

#create df for selected questions to use in test set later
df_test_additional = df[df['question'].isin(selected_questions)].reset_index(drop=True)

#remove selected questions from training set
train_df_with_subset_removed = df[~df['question'].isin(selected_questions)].reset_index(drop=True)

##### finalize training data

In [15]:
#check numbers
print(train_df_with_subset_removed.shape)
print(train_df_with_subset_removed.question.nunique())
print('check numbers:', df['question'].nunique() - n_samples_move_from_train_to_test == train_df_with_subset_removed.question.nunique())

(123920, 10)
10003
check numbers: True


###### write question in SFT format (sft_input column)

In [16]:
#questions have different numbers of choices
train_df_with_subset_removed['n_choices'] = train_df_with_subset_removed.apply(lambda row: len(row['options']), axis=1).tolist()
print(set(train_df_with_subset_removed['n_choices']))
# train_df_with_subset_removed[train_df_with_subset_removed['n_choices'] == 3].head()#['options']

{3, 4, 5, 6, 7, 8, 9, 10}


In [17]:
letters = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J']


def write_sft_input(row):
    #write string for options
    options = row['options']
    
    options_string = ""
    for i, option in enumerate(options):
        options_string += f"\n{letters[i]}. {option}"

    #write sft input
    question = row['question']
    sft_input = f'{question}{options_string}'

    return sft_input

In [18]:
#format input and output for all rows
train_df_with_subset_removed['sft_input'] = train_df_with_subset_removed.apply(write_sft_input, axis=1)
#sft_output is train_df_with_subset_removed['cot_content']

###### check lengths of examples

In [19]:
#check lengths: sft_input + sft_output < 2048*3
train_df_with_subset_removed['total_length'] = train_df_with_subset_removed['sft_input'].str.len() + train_df_with_subset_removed['cot_content'].str.len()
train_df_with_subset_removed['total_length'].describe()
print(train_df_with_subset_removed['total_length'].describe())

#keep only rows where total_length < 2048*3
max_length = 2048*3
train_df_final = train_df_with_subset_removed[train_df_with_subset_removed['total_length'] < max_length].reset_index(drop=True)
print(train_df_final['total_length'].describe())

count    123920.000000
mean       1859.001541
std         878.859056
min         192.000000
25%        1197.000000
50%        1732.000000
75%        2385.000000
max        8737.000000
Name: total_length, dtype: float64
count    123836.000000
mean       1855.643157
std         869.508099
min         192.000000
25%        1197.000000
50%        1732.000000
75%        2384.000000
max        6134.000000
Name: total_length, dtype: float64


##### save training data to json

In [20]:
#lood at example
idx = 11
print('sft_input:')
print(train_df_final.iloc[idx]['sft_input'])

print('\nsft_output:')
print(train_df_final.iloc[idx]['cot_content'])

sft_input:
Typical advertising regulatory bodies suggest, for example that adverts must not: encourage _________, cause unnecessary ________ or _____, and must not cause _______ offence.
A. Safe practices, Fear, Jealousy, Trivial
B. Unsafe practices, Distress, Joy, Trivial
C. Safe practices, Wants, Jealousy, Trivial
D. Safe practices, Distress, Fear, Trivial
E. Unsafe practices, Wants, Jealousy, Serious
F. Safe practices, Distress, Jealousy, Serious
G. Safe practices, Wants, Fear, Serious
H. Unsafe practices, Wants, Fear, Trivial
I. Unsafe practices, Distress, Fear, Serious

sft_output:
First, we need to consider the typical guidelines for advertising regulatory bodies. They usually aim to ensure that advertisements do not have a negative impact on the public.

The first part of the statement is "encourage _______ practices". Since the goal is to prevent harm, it's reasonable to assume that regulatory bodies would want to discourage "unsafe" practices rather than "safe" ones.

Next, we

In [21]:
### create json file

#format data for sft
data = []

for idx in tqdm(range(train_df_final.shape[0])):
    row = train_df_final.iloc[idx]
    item = {
        "instruction": row["sft_input"],
        "input": "",
        "output": row["cot_content"]
    }
    data.append(item)

#save to JSON file
with open("data/mmluprocot.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print("Final # of samples in json file:", len(data))
print("Complete!")

  0%|                                                              | 0/123836 [00:00<?, ?it/s]

100%|██████████████████████████████████████████████| 123836/123836 [00:03<00:00, 31524.92it/s]


Final # of samples in json file: 123836
Complete!


## create test dataset: combine test_df + samples removed from training set

In [22]:
#combine test_df + samples removed from training set
test_df_final = pd.concat([test_df, df_test_additional], ignore_index=True)

#keep only unique questions
test_df_final = test_df_final.drop_duplicates(subset='question')

In [23]:
#check
print('# unique questions in test set:', test_df_final.question.nunique())
print('check:', test_df.question.nunique() + n_samples_move_from_train_to_test == test_df_final.question.nunique())

# unique questions in test set: 567
check: True


In [24]:
#check that there is no train / test overlap
qs_unique_train = set(train_df_final['question'])
qs_unique_test = set(test_df_final['question'])

overlap = qs_unique_train.intersection(qs_unique_test)
print(len(overlap))
print(overlap)

0
set()


In [25]:
#format sft_input for all rows
test_df_final['sft_input'] = test_df_final.apply(write_sft_input, axis=1)

In [26]:
test_df_final.head()

,question_id,question,options,answer,answer_index,category,src,cot_id,cot_content,extracted_answer,sft_input
0,0,The symmetric group $S_n$ has $\n\factorial{n}...,"[0, 30, 3, 10, 12, 50, 2, 100, 20, 5]",A,0,math,cot_lib-abstract_algebra,0,"We are given a problem with two parts, but the...",NaN,The symmetric group $S_n$ has $\n\factorial{n}...
9,1,Let V be the set of all real polynomials p(x)....,[ST + TS is the identity map of V onto itself....,H,7,math,cot_lib-college_mathematics,0,Given transformations T and S defined on V by:...,NaN,Let V be the set of all real polynomials p(x)....
25,2,Let A be the set of all ordered pairs of integ...,"[-5, 0, -3, -7, -4, -6, -1, -2, -9]",E,4,math,cot_lib-college_mathematics,0,We are given the equation 7m + 12n = 22. We ne...,NaN,Let A be the set of all ordered pairs of integ...
29,3,A tank initially contains a salt solution of 3...,"[3 + e^-2, 2 - e^-4, 2 - e^-2, 3 + e^-4, 2 + e...",I,8,math,cot_lib-college_mathematics,0,"Initially, there are 3 grams of salt in the ta...",NaN,A tank initially contains a salt solution of 3...
31,4,A total of 30 players will play basketball at ...,"[Multiply 5 by 5 to find 25 teams., Divide 30 ...",B,1,math,cot_lib-elementary_mathematics,0,1. We know there are 30 players in total.\n2. ...,NaN,A total of 30 players will play basketball at ...


In [27]:
#create new question_id's
test_df_final['question_id_new'] = list(range(1, test_df_final.shape[0]+1))
test_df_final['question_id_new'] = test_df_final['question_id_new'].astype(str) + '-' + test_df_final['src']

In [28]:
### MOVE DATA BY HAND INTO CORRECT FOLDER : learn-better/evolm/evaluation/cot-eval-harness/data/cooked

### create jsonl file

data = []

# Write JSONL file
with open("data/mmluprocot.jsonl", "w", encoding="utf-8") as f:
    for idx in tqdm(range(test_df_final.shape[0])):
        row = test_df_final.iloc[idx]
        item = {
            "id": str(row["question_id_new"]),
            "problem": str(row["sft_input"]),
            "gt_solution": str(row["answer"]),
            "gt_answer": str(row["answer"])
        }
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Final # of samples in json file:", len(data))
print("Complete!")

100%|████████████████████████████████████████████████████| 567/567 [00:00<00:00, 19077.41it/s]

Final # of samples in json file: 0
Complete!
